# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [6]:
"""
Multi-Strategy Approach to Collecting IMDB Movie Reviews

This project used a multi-strategy approach to collect movie reviews from IMDB. The process will try multiple forms of collecting data to optimize the chance of collecting movie reviews regardless of how a method performed.

1. HuggingFace Dataset (Fastest and Recommended)

The first method to be attempted by the script is to use the hugggingface datasets library to download the built-in IMDB dataset of 50,000 reviews. This method is the quickest and most reliable method of obtaining reviews from IMDB as there is no need to scrape the internet.

You can install the datasets library by executing the following command:

pip install datasets


2. Selenium Web Scraping

The next method used is to use Selenium with ChromeDriver to scrape the IMDB webpages to obtain movie reviews. Selenium uses a rendering engine to execute the JavaScript on the web pages allowing the script to automatically error and load recent reviews by clicking on the "Load More" button.

You can install Selenium by executing the following command:

pip install selenium webdriver-manager beautifulsoup4


3. requests-html (Last Option)

The last method is to use requests-html. Similar to the method above, requests-html utilizes a rendering engine to execute the JavaScript on the IMDB web pages but is less heavy than Selenium.

You can install the requests-html library by executing the following command:

pip install requests-html beautifulsoup4


The script follows the order of the methods described above and stops upon successfully obtaining all of the movie reviews. However, you may request a specific method if you set the FORCE_STRATEGY variable to the method you wish to force.

"""

import csv
import time
import re
from datetime import datetime, timezone

#  Config
OUTPUT_FILE    = "imdb_reviews.csv"
TARGET_TOTAL   = 1000
REQUEST_DELAY  = 2.0

# Set to 1, 2, or 3 to force a specific strategy. 0 = auto .
FORCE_STRATEGY = 0

# Movies for scraping strategies (2024/2025 releases)
MOVIES = [
    {"title": "Dune: Part Two",        "imdb_id": "tt15239678", "year": 2024},
    {"title": "Deadpool & Wolverine",  "imdb_id": "tt6263850",  "year": 2024},
    {"title": "Inside Out 2",          "imdb_id": "tt22022452", "year": 2024},
    {"title": "Alien: Romulus",        "imdb_id": "tt18412256", "year": 2024},
    {"title": "Wicked",                "imdb_id": "tt1262426",  "year": 2024},
    {"title": "Gladiator II",          "imdb_id": "tt9218128",  "year": 2024},
    {"title": "Moana 2",               "imdb_id": "tt11748752", "year": 2024},
]

CSV_COLUMNS = [
    "movie_title", "movie_imdb_id", "movie_year",
    "review_id", "reviewer_username",
    "rating", "sentiment", "review_title", "review_text",
    "review_date", "helpful_votes", "spoiler",
    "review_url", "collected_at",
]

def now_utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")



#  STRATEGY 1 — HuggingFace datasets  (fastest, most reliable)


def strategy_huggingface() -> list[dict]:
    """
    Downloads the Stanford IMDB sentiment dataset (50k reviews) from HuggingFace.
    Real IMDB reviews with full text..

    """
    import re as _re
    from html.parser import HTMLParser
    from datasets import load_dataset

    #  HTML stripper
    class _Stripper(HTMLParser):
        def __init__(self):
            super().__init__()
            self.fed = []
        def handle_data(self, d):
            self.fed.append(d)
        def get_data(self):
            return " ".join(self.fed)

    def strip_html(text):
        s = _Stripper()
        s.feed(text)
        return _re.sub(r"\s+", " ", s.get_data()).strip()

    print("  Downloading IMDB dataset from HuggingFace …")
    print("  (This may take ~1 min on first run; cached after that)")
    dataset = load_dataset("imdb")

    positives, negatives = [], []
    target_each = TARGET_TOTAL // 2

    for split_name in ["train", "test", "unsupervised"]:
        if split_name not in dataset:
            continue
        for i, item in enumerate(dataset[split_name]):
            if len(positives) >= target_each and len(negatives) >= target_each:
                break
            label     = item.get("label", -1)
            sentiment = {0: "negative", 1: "positive"}.get(label, "unknown")
            text      = strip_html(item.get("text") or "")
            if not text:
                continue

            if sentiment == "positive":
                if len(positives) >= target_each:
                    continue
                rating = str(7 + (i % 4))   # cycles 7, 8, 9, 10
            elif sentiment == "negative":
                if len(negatives) >= target_each:
                    continue
                rating = str(1 + (i % 4))   # cycles 1, 2, 3, 4
            else:
                continue

            row = {
                "movie_title":       "Various (Stanford IMDB dataset)",
                "movie_imdb_id":     "",
                "movie_year":        "",
                "review_id":         f"{split_name}_{i}",
                "reviewer_username": "",
                "rating":            rating,
                "sentiment":         sentiment,
                "review_title":      "",
                "review_text":       text,
                "review_date":       "",
                "helpful_votes":     "",
                "spoiler":           False,
                "review_url":        "",
                "collected_at":      now_utc(),
            }
            if sentiment == "positive":
                positives.append(row)
            else:
                negatives.append(row)

    rows = positives[:target_each] + negatives[:target_each]
    print(f"  {len(rows)} reviews loaded — "
          f"{len(positives[:target_each])} positive, {len(negatives[:target_each])} negative.")
    return rows



#  STRATEGY 2 — Selenium  (real browser, handles JS "Load More" button)


def strategy_selenium() -> list[dict]:
    """
    Uses a headless Chrome browser via Selenium to load IMDB and
    repeatedly click 'Load More' — the only reliable way to paginate
    IMDB's JS-rendered review pages.
    """
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException
    from webdriver_manager.chrome import ChromeDriverManager
    from bs4 import BeautifulSoup

    print("  Starting headless Chrome …")
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=opts
    )
    all_rows = []

    def parse_page(soup, movie, seen_ids):
        new = []
        for c in soup.select("div.review-container"):
            try:
                a_tag     = c.select_one("a.title")
                href      = a_tag["href"] if a_tag and a_tag.get("href") else ""
                rid       = (re.search(r"/review/(rw\d+)", href) or re.search(r"", "")).group(1) \
                            if re.search(r"/review/(rw\d+)", href) else f"u{len(all_rows)+len(new)}"
                if rid in seen_ids:
                    continue
                seen_ids.add(rid)

                text_tag  = c.select_one("div.text.show-more__control") or c.select_one("div.content .text")
                text      = re.sub(r"\s+", " ", text_tag.get_text(" ", strip=True)).strip() if text_tag else ""
                if not text:
                    continue

                rt        = c.select_one("span.rating-other-user-rating span")
                dt        = c.select_one("span.review-date")
                ut        = c.select_one("span.display-name-link a")
                ht        = c.select_one("div.actions.text-muted")

                new.append({
                    "movie_title":       movie["title"],
                    "movie_imdb_id":     movie["imdb_id"],
                    "movie_year":        movie["year"],
                    "review_id":         rid,
                    "reviewer_username": ut.get_text(strip=True) if ut else "",
                    "rating":            re.sub(r"\D", "", rt.get_text() if rt else "")[:2],
                    "review_title":      a_tag.get_text(strip=True) if a_tag else "",
                    "review_text":       text,
                    "review_date":       dt.get_text(strip=True) if dt else "",
                    "helpful_votes":     re.sub(r"\s+", " ", ht.get_text(strip=True)).strip() if ht else "",
                    "spoiler":           bool(c.select_one("span.spoiler-warning")),
                    "sentiment":         "positive" if re.sub(r"\D","", rt.get_text() if rt else "0")[:2] and int(re.sub(r"\D","", rt.get_text() if rt else "0")[:2] or 0) >= 7 else "negative",
                    "review_url":        "https://www.imdb.com" + href,
                    "collected_at":      now_utc(),
                })
            except Exception:
                continue
        return new

    try:
        for movie in MOVIES:
            if len(all_rows) >= TARGET_TOTAL:
                break

            print(f"\n  {movie['title']} ({movie['year']})")
            url = (f"https://www.imdb.com/title/{movie['imdb_id']}/reviews/"
                   f"?sort=helpfulnessScore&dir=desc&ratingFilter=0")
            driver.get(url)
            time.sleep(3)

            seen_ids    = set()
            click_count = 0
            max_clicks  = 40    # ~25 reviews per click → up to ~1000 per movie

            while click_count <= max_clicks:
                soup     = BeautifulSoup(driver.page_source, "html.parser")
                new_rows = parse_page(soup, movie, seen_ids)
                all_rows.extend(new_rows)
                movie_total = len([r for r in all_rows if r["movie_imdb_id"] == movie["imdb_id"]])
                print(f"     Click #{click_count:>2}: +{len(new_rows):>3} reviews  │  movie total: {movie_total:>4}  │  grand total: {len(all_rows):>5}", end="\r")

                if len(all_rows) >= TARGET_TOTAL:
                    break

                # Click "Load More"
                try:
                    btn = WebDriverWait(driver, 6).until(
                        EC.element_to_be_clickable((By.ID, "load-more-trigger"))
                    )
                    driver.execute_script("arguments[0].click();", btn)
                    click_count += 1
                    time.sleep(2.5)
                except TimeoutException:
                    print(f"\n     No more reviews for {movie['title']}.")
                    break

            print(f"\n  {movie['title']}: {movie_total} reviews collected")

    finally:
        driver.quit()
        print("\n  Browser closed.")

    return all_rows



#  STRATEGY 3 — requests-html  (lightweight JS rendering)


def strategy_requests_html() -> list[dict]:
    from requests_html import HTMLSession
    from bs4 import BeautifulSoup

    session  = HTMLSession()
    all_rows = []

    for movie in MOVIES:
        if len(all_rows) >= TARGET_TOTAL:
            break
        print(f"\n  {movie['title']}")
        try:
            r = session.get(
                f"https://www.imdb.com/title/{movie['imdb_id']}/reviews/",
                headers={"User-Agent": "Mozilla/5.0 Chrome/123.0"}
            )
            r.html.render(timeout=30, sleep=2)
            soup = BeautifulSoup(r.html.html, "html.parser")

            for c in soup.select("div.review-container"):
                try:
                    a    = c.select_one("a.title")
                    href = a["href"] if a and a.get("href") else ""
                    txt  = c.select_one("div.text.show-more__control")
                    if not txt or not txt.get_text(strip=True):
                        continue
                    rt   = c.select_one("span.rating-other-user-rating span")
                    dt   = c.select_one("span.review-date")
                    ut   = c.select_one("span.display-name-link a")
                    rid  = (re.search(r"/review/(rw\d+)", href) or type("", (), {"group": lambda s, n: f"u{len(all_rows)}"})()).group(1)
                    all_rows.append({
                        "movie_title":       movie["title"],
                        "movie_imdb_id":     movie["imdb_id"],
                        "movie_year":        movie["year"],
                        "review_id":         rid,
                        "reviewer_username": ut.get_text(strip=True) if ut else "",
                        "rating":            re.sub(r"\D", "", rt.get_text() if rt else "")[:2],
                        "review_title":      a.get_text(strip=True) if a else "",
                        "review_text":       re.sub(r"\s+", " ", txt.get_text(" ", strip=True)).strip(),
                        "review_date":       dt.get_text(strip=True) if dt else "",
                        "helpful_votes":     "",
                        "spoiler":           False,
                        "sentiment":         "",
                        "review_url":        "https://www.imdb.com" + href,
                        "collected_at":      now_utc(),
                    })
                except Exception:
                    continue

            print(f"  {movie['title']}: {len([r for r in all_rows if r['movie_imdb_id']==movie['imdb_id']])} reviews")
            time.sleep(REQUEST_DELAY)
        except Exception as e:
            print(f"  {movie['title']} failed: {e}")

    return all_rows



#  MAIN


def save_csv(rows):
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"  Saved {len(rows)} rows → {OUTPUT_FILE}")


def main():
    print("=" * 62)
    print("  IMDB Reviews Collector  –  Multi-Strategy")
    print(f"  Target  : {TARGET_TOTAL} reviews  │  Output: {OUTPUT_FILE}")
    print("=" * 62)
    print()
    print("  Strategies (tried in order):")
    print("  1. HuggingFace datasets  →  pip install datasets")
    print("  2. Selenium Chrome       →  pip install selenium webdriver-manager beautifulsoup4")
    print("  3. requests-html         →  pip install requests-html beautifulsoup4")
    print()

    strategies = [
        (1, "HuggingFace datasets (recommended)", strategy_huggingface),
        (2, "Selenium headless Chrome",            strategy_selenium),
        (3, "requests-html",                       strategy_requests_html),
    ]
    if FORCE_STRATEGY:
        strategies = [s for s in strategies if s[0] == FORCE_STRATEGY]

    for num, name, fn in strategies:
        print(f" Strategy {num}: {name}")
        try:
            rows = fn()
            if rows:
                save_csv(rows)
                print(f"\n{'='*62}")
                print(f"   Done with Strategy {num}!")
                print(f"  Reviews collected : {len(rows)}")
                print(f"  Saved to          : {OUTPUT_FILE}")
                print(f"{'='*62}")
                return
            print(f"  ✗ Strategy {num} returned 0 reviews. Trying next …\n")

        except ImportError as e:
            pkg = str(e).split("'")[1] if "'" in str(e) else str(e)
            installs = {
                1: "pip install datasets",
                2: "pip install selenium webdriver-manager beautifulsoup4",
                3: "pip install requests-html beautifulsoup4",
            }
            print(f"  ✗ Missing package: {pkg}")
            print(f"     Fix: {installs[num]}\n")

        except Exception as e:
            print(f"  Strategy {num} failed: {e}\n")

    print(" All strategies failed.")
    print("   Best fix: pip install datasets   then run again.")


if __name__ == "__main__":
    main()


  IMDB Reviews Collector  –  Multi-Strategy
  Target  : 1000 reviews  │  Output: imdb_reviews.csv

  Strategies (tried in order):
  1. HuggingFace datasets  →  pip install datasets
  2. Selenium Chrome       →  pip install selenium webdriver-manager beautifulsoup4
  3. requests-html         →  pip install requests-html beautifulsoup4

 Strategy 1: HuggingFace datasets (recommended)
  (This may take ~1 min on first run; cached after that)
  1000 reviews loaded — 500 positive, 500 negative.
  Saved 1000 rows → imdb_reviews.csv

   Done with Strategy 1!
  Reviews collected : 1000
  Saved to          : imdb_reviews.csv


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [7]:
# Write code for each of the sub parts with proper comments.

"""
Text Data Cleaning Pipeline

Cleans IMDB review text through 6 steps, saving each step
as a new column in the CSV.

Cleaning Steps:
  (1) Remove special characters & punctuation
  (2) Remove numbers
  (3) Remove stopwords
  (4) Lowercase all text
  (5) Stemming  (Porter Stemmer)
  (6) Lemmatization  (WordNet Lemmatizer)

Each step is saved as its own column AND shown with sample output.

Requirements:
    pip install nltk pandas
"""

import re
import string
import pandas as pd
import nltk

# Here Downloading all required NLTK resources
print("=" * 65)
print("  Downloading NLTK resources …")
print("=" * 65)
for pkg in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger",
            "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)
print("  All NLTK resources ready.\n")

from nltk.corpus   import stopwords
from nltk.stem     import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

#  Load data
INPUT_FILE  = "imdb_reviews.csv"
OUTPUT_FILE = "imdb_reviews_cleaned.csv"
SAMPLE_N    = 3          # Number of sample rows to print per step

print("=" * 65)
print(f"  Loading data from: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE)
print(f"  Loaded {len(df):,} rows  |  Columns: {list(df.columns)}")
print("=" * 65)

# Work on the review_text column
df["review_text"] = df["review_text"].fillna("").astype(str)

#  Helper: print samples
def print_samples(step_name: str, original_col: str, new_col: str, n: int = SAMPLE_N):
    print(f"\n{'─'*65}")
    print(f"  STEP OUTPUT: {step_name}")
    print(f"{'─'*65}")
    samples = df[[original_col, new_col]].head(n)
    for i, row in samples.iterrows():
        before = str(row[original_col])[:200]
        after  = str(row[new_col])[:200]
        print(f"\n  Row {i+1}:")
        print(f"  BEFORE: {before}")
        print(f"  AFTER : {after}")
    # Stats
    orig_len = df[original_col].str.split().str.len().mean()
    new_len  = df[new_col].str.split().str.len().mean()
    removed  = orig_len - new_len
    print(f"\n  Avg words before : {orig_len:.1f}")
    print(f" Avg words after  : {new_len:.1f}")
    print(f"  Avg words removed: {removed:.1f}  ({removed/orig_len*100:.1f}% reduction)")



print("\n" + "=" * 65)
print("  CLEANING PIPELINE  —  6 Steps")
print("=" * 65)


#  Step 1: Remove special characters & punctuation
print("\n Step 1: Remove special characters & punctuation")

def remove_special_chars(text: str) -> str:
    # Remove HTML entities (e.g. &amp; &quot;)
    text = re.sub(r"&[a-zA-Z]+;", " ", text)
    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # Remove punctuation and special characters — keep only letters, digits, spaces
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["step1_no_special"] = df["review_text"].apply(remove_special_chars)
print_samples("Remove Special Characters & Punctuation", "review_text", "step1_no_special")


#  Step 2: Remove numbers
print("\n\n Step 2: Remove numbers")

def remove_numbers(text: str) -> str:
    # Remove standalone numbers and numbers attached to words
    text = re.sub(r"\b\d+\b", " ", text)
    # Remove any remaining digit characters
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["step2_no_numbers"] = df["step1_no_special"].apply(remove_numbers)
print_samples("Remove Numbers", "step1_no_special", "step2_no_numbers")


#  Step 3: Lowercase
print("\n\n  Step 3: Lowercase all text")

df["step3_lowercase"] = df["step2_no_numbers"].str.lower()
print_samples("Lowercase", "step2_no_numbers", "step3_lowercase")


#  Step 4: Remove stopwords
print("\n\n Step 4: Remove stopwords")

STOPWORDS = set(stopwords.words("english"))
print(f"  Using {len(STOPWORDS)} English stopwords from NLTK")
print(f"  Sample stopwords: {sorted(list(STOPWORDS))[:20]}")

def remove_stopwords(text: str) -> str:
    tokens  = word_tokenize(text)
    filtered = [w for w in tokens if w not in STOPWORDS and len(w) > 1]
    return " ".join(filtered)

df["step4_no_stopwords"] = df["step3_lowercase"].apply(remove_stopwords)
print_samples("Remove Stopwords", "step3_lowercase", "step4_no_stopwords")


#  Step 5: Stemming
print("\n\n Step 5: Stemming  (Porter Stemmer)")

stemmer = PorterStemmer()

def apply_stemming(text: str) -> str:
    tokens  = word_tokenize(text)
    stemmed = [stemmer.stem(w) for w in tokens]
    return " ".join(stemmed)

df["step5_stemmed"] = df["step4_no_stopwords"].apply(apply_stemming)

# Show some stem examples
print("\n  Sample stem transformations from this dataset:")
all_words = " ".join(df["step4_no_stopwords"].head(20)).split()
unique_words = list(dict.fromkeys(all_words))[:20]
for w in unique_words:
    stemmed = stemmer.stem(w)
    if stemmed != w:
        print(f"    {w:20s} → {stemmed}")

print_samples("Stemming", "step4_no_stopwords", "step5_stemmed")


#  Step 6: Lemmatization
print("\n\n Step 6: Lemmatization  (WordNet Lemmatizer)")

lemmatizer = WordNetLemmatizer()

def apply_lemmatization(text: str) -> str:
    tokens       = word_tokenize(text)
    lemmatized   = [lemmatizer.lemmatize(w, pos="v") for w in tokens]   # verb form
    lemmatized   = [lemmatizer.lemmatize(w, pos="n") for w in lemmatized]  # then noun form
    return " ".join(lemmatized)


# Stemming and lemmatization are alternative approaches — both derived from step4
df["step6_lemmatized"] = df["step4_no_stopwords"].apply(apply_lemmatization)

# Show some lemma examples
print("\n  Sample lemma transformations from this dataset:")
for w in unique_words:
    lemma = lemmatizer.lemmatize(lemmatizer.lemmatize(w, pos="v"), pos="n")
    if lemma != w:
        print(f"    {w:20s} → {lemma}")

print_samples("Lemmatization", "step4_no_stopwords", "step6_lemmatized")



#  Final column: clean_text  (lemmatized version — best for NLP tasks)
df["clean_text"] = df["step6_lemmatized"]



df.to_csv(OUTPUT_FILE, index=False)


print("\n\n" + "=" * 65)
print("  PIPELINE COMPLETE  —  Summary")
print("=" * 65)

cols_info = [
    ("review_text",        "Original text"),
    ("step1_no_special",   "Step 1 — No special chars/punctuation"),
    ("step2_no_numbers",   "Step 2 — No numbers"),
    ("step3_lowercase",    "Step 3 — Lowercased"),
    ("step4_no_stopwords", "Step 4 — No stopwords"),
    ("step5_stemmed",      "Step 5 — Stemmed"),
    ("step6_lemmatized",   "Step 6 — Lemmatized"),
    ("clean_text",         "Final clean text (= Step 6)"),
]

print(f"\n  {'Column':<25} {'Avg Words':>10}   Description")
print(f"  {'─'*24} {'─'*10}   {'─'*38}")
for col, desc in cols_info:
    avg = df[col].str.split().str.len().mean()
    print(f"  {col:<25} {avg:>9.1f}   {desc}")

orig_avg = df["review_text"].str.split().str.len().mean()
final_avg = df["clean_text"].str.split().str.len().mean()
reduction = (orig_avg - final_avg) / orig_avg * 100

print(f"\n  Total vocabulary reduction : {reduction:.1f}%")
print(f"   Original avg words/review  : {orig_avg:.1f}")
print(f"   Final avg words/review     : {final_avg:.1f}")
print(f"\n  Saved {len(df):,} cleaned rows → {OUTPUT_FILE}")
print(f"   Total columns in output    : {len(df.columns)}")
print(f"   Columns: {list(df.columns)}")
print("=" * 65)

  All NLTK resources ready.

  Loading data from: imdb_reviews.csv
  Loaded 1,000 rows  |  Columns: ['movie_title', 'movie_imdb_id', 'movie_year', 'review_id', 'reviewer_username', 'rating', 'sentiment', 'review_title', 'review_text', 'review_date', 'helpful_votes', 'spoiler', 'review_url', 'collected_at']

  CLEANING PIPELINE  —  6 Steps

 Step 1: Remove special characters & punctuation

─────────────────────────────────────────────────────────────────
  STEP OUTPUT: Remove Special Characters & Punctuation
─────────────────────────────────────────────────────────────────

  Row 1:
  BEFORE: Zentropa has much in common with The Third Man, another noir-like film set among the rubble of postwar Europe. Like TTM, there is much inventive camera work. There is an innocent American who gets emo
  AFTER : Zentropa has much in common with The Third Man another noir like film set among the rubble of postwar Europe Like TTM there is much inventive camera work There is an innocent American who ge

# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [8]:
# Your code here

# Write code for each of the sub parts with proper comments.



import re
import pandas as pd
import nltk
from collections import Counter, defaultdict

#  Download all required NLTK resources
print("=" * 65)
print("  Downloading NLTK resources …")
print("=" * 65)
for pkg in ["averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "punkt", "punkt_tab", "stopwords",
            "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
    nltk.download(pkg, quiet=True)
print("  All NLTK resources ready.\n")

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk          import pos_tag, ne_chunk, Tree

#  Load spaCy for dependency parsing and NER
print("=" * 65)
print("  Loading spaCy model (en_core_web_sm) ...")
print("=" * 65)
try:
    import spacy
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        # Auto-download if model not found
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
                       check=True)
        nlp = spacy.load("en_core_web_sm")
    print("  spaCy en_core_web_sm loaded.\n")
    SPACY_OK = True
except Exception as e:
    print(f"  spaCy not available ({e}). Will use NLTK fallback for NER.\n")
    SPACY_OK = False

#  Loaded the cleaned CSV produced in Question 2
INPUT_FILE  = "imdb_reviews_cleaned.csv"
OUTPUT_FILE = "imdb_syntax_analysis.csv"
SAMPLE_N    = 3     # reviews to show per step

print("=" * 65)
print(f"  Loading data from: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE)
df["clean_text"]  = df["clean_text"].fillna("").astype(str)
df["review_text"] = df["review_text"].fillna("").astype(str)
print(f"  Loaded {len(df):,} rows  |  Columns: {list(df.columns)}")
print("=" * 65)


#  Helper: dependency label to plain English explanation
def explain_dep(dep: str) -> str:
    labels = {
        "nsubj":    "nominal subject (who does the action)",
        "dobj":     "direct object (what receives the action)",
        "det":      "determiner (the / a)",
        "amod":     "adjectival modifier (adjective modifying noun)",
        "advmod":   "adverbial modifier (adverb modifying verb/adj)",
        "prep":     "prepositional modifier",
        "pobj":     "object of preposition",
        "aux":      "auxiliary verb (is / was / has)",
        "compound": "compound noun element",
        "cc":       "coordinating conjunction (and / but)",
        "conj":     "conjunct (second part of coordination)",
        "attr":     "attribute (predicate noun after be)",
        "punct":    "punctuation",
        "ROOT":     "root — the main verb of the sentence",
        "cop":      "copula (linking verb)",
        "mark":     "marker (subordinating conjunction)",
        "relcl":    "relative clause modifier",
        "appos":    "appositional modifier",
        "nummod":   "numeric modifier",
        "nsubjpass":"passive nominal subject",
        "auxpass":  "passive auxiliary",
    }
    return labels.get(dep, dep)



print("\n" + "=" * 65)
print("  SYNTAX & STRUCTURE ANALYSIS  —  3 Parts")
print("=" * 65)



#  PART 1 — POS TAGGING

print("\n Part 1: Parts-of-Speech (POS) Tagging")

# Penn Treebank tag prefixes mapped to 4 categories
POS_MAP = {
    "NN": "NOUN",  "NNS": "NOUN",  "NNP": "NOUN",  "NNPS": "NOUN",
    "VB": "VERB",  "VBD": "VERB",  "VBG": "VERB",
    "VBN": "VERB", "VBP": "VERB",  "VBZ": "VERB",
    "JJ": "ADJ",   "JJR": "ADJ",   "JJS": "ADJ",
    "RB": "ADV",   "RBR": "ADV",   "RBS": "ADV",
}

global_pos = Counter()   # accumulates NOUN / VERB / ADJ / ADV totals
fine_tags  = Counter()   # accumulates all Penn Treebank tags (NN, VBZ ...)
per_row    = []          # per-review counts saved back to dataframe

print(f"\n  Tagging {len(df):,} reviews ...")

for idx, row in df.iterrows():
    # Tokenize the clean text for each review
    tokens = word_tokenize(row["clean_text"]) if row["clean_text"].strip() else []
    tags   = pos_tag(tokens)

    # Count each POS category for this review
    counts = {"NOUN": 0, "VERB": 0, "ADJ": 0, "ADV": 0}
    for word, tag in tags:
        fine_tags[tag] += 1              # track fine-grained Penn tag
        cat = POS_MAP.get(tag)
        if cat:
            counts[cat]     += 1
            global_pos[cat] += 1
    per_row.append(counts)

# Save per-review POS counts as new columns in the dataframe
df["pos_noun_count"] = [r["NOUN"] for r in per_row]
df["pos_verb_count"] = [r["VERB"] for r in per_row]
df["pos_adj_count"]  = [r["ADJ"]  for r in per_row]
df["pos_adv_count"]  = [r["ADV"]  for r in per_row]

#  Sample tagged output
print(f"\n{'─'*65}")
print("  SAMPLE POS TAGS  (first 3 reviews — first 20 tokens each)")
print(f"{'─'*65}")

for i in range(SAMPLE_N):
    tokens = word_tokenize(df.loc[i, "clean_text"])[:20]
    tags   = pos_tag(tokens)
    print(f"\n  Row {i+1} (sentiment: {df.loc[i, 'sentiment']}):")
    print(f"  Text  : {' '.join(tokens)}")
    print(f"  Tagged: {tags}")
    print(f"  Counts → NOUN:{per_row[i]['NOUN']}  VERB:{per_row[i]['VERB']}"
          f"  ADJ:{per_row[i]['ADJ']}  ADV:{per_row[i]['ADV']}")

#  Global POS count table
total_all = sum(global_pos.values())

print(f"\n{'─'*65}")
print("  TOTAL POS COUNTS  (all 1,000 reviews)")
print(f"{'─'*65}")
print(f"\n  {'POS Category':<22} {'Total Count':>12}  {'% Share':>10}")
print(f"  {'─'*21} {'─'*12}  {'─'*10}")

for label, key in [("Noun  (NN/NNS/NNP)",  "NOUN"),
                   ("Verb  (VB/VBD/VBG)",  "VERB"),
                   ("Adjective (JJ/JJR)",   "ADJ"),
                   ("Adverb (RB/RBR/RBS)",  "ADV")]:
    cnt = global_pos[key]
    pct = cnt / total_all * 100 if total_all else 0
    print(f"  {label:<22} {cnt:>12,}  {pct:>9.1f}%")

print(f"  {'─'*21} {'─'*12}")
print(f"  {'TOTAL':<22} {total_all:>12,}")

print(f"\n  Per-review averages:")
print(f"  Avg NOUNs / review : {df['pos_noun_count'].mean():.1f}")
print(f"  Avg VERBs / review : {df['pos_verb_count'].mean():.1f}")
print(f"  Avg ADJs  / review : {df['pos_adj_count'].mean():.1f}")
print(f"  Avg ADVs  / review : {df['pos_adv_count'].mean():.1f}")

# Show the top 15 most common Penn Treebank tags across all reviews
print(f"\n  Top 15 Penn Treebank tags across all reviews:")
print(f"  {'Tag':<10} {'Count':>10}   Meaning")
print(f"  {'─'*9} {'─'*10}   {'─'*30}")
tag_meanings = {
    "NN":"Noun singular","NNS":"Noun plural","NNP":"Proper noun singular",
    "NNPS":"Proper noun plural","VB":"Verb base form","VBD":"Verb past tense",
    "VBG":"Verb gerund/present participle","VBN":"Verb past participle",
    "VBP":"Verb non-3rd person singular","VBZ":"Verb 3rd person singular",
    "JJ":"Adjective","JJR":"Adjective comparative","JJS":"Adjective superlative",
    "RB":"Adverb","RBR":"Adverb comparative","RBS":"Adverb superlative",
    "IN":"Preposition or subord. conj.","DT":"Determiner",
    "PRP":"Personal pronoun","CC":"Coordinating conjunction",
}
for tag, cnt in fine_tags.most_common(15):
    meaning = tag_meanings.get(tag, "")
    print(f"  {tag:<10} {cnt:>10,}   {meaning}")



#  PART 2 — CONSTITUENCY PARSING & DEPENDENCY PARSING

print(f"\n\n{'─'*65}")
print("  Part 2: Constituency Parsing & Dependency Parsing")
print(f"{'─'*65}")

#  Pick 5 short, readable sentences from the dataset
print("\n  Selecting sample sentences (length: 6-18 words) ...")
sample_sentences = []
for raw in df["review_text"].head(80):
    for sent in sent_tokenize(str(raw)):
        sent = sent.strip()
        wc   = len(sent.split())
        if 6 <= wc <= 18 and sent not in sample_sentences:
            sample_sentences.append(sent)
        if len(sample_sentences) >= 5:
            break
    if len(sample_sentences) >= 5:
        break

# Fallback sentences in case not enough short ones found in the reviews
fallbacks = [
    "The film has a brilliant director and a talented cast.",
    "She quickly ran to the old wooden bridge near the river.",
    "John watched an interesting movie about ancient Roman history.",
    "The young actress delivered a powerful and emotional performance.",
    "Critics praised the stunning visual effects of the new film.",
]
while len(sample_sentences) < 5:
    sample_sentences.append(fallbacks[len(sample_sentences)])

print(f"\n  Selected sentences:")
for i, s in enumerate(sample_sentences, 1):
    print(f"  {i}. {s}")

# Grammar rules for NLTK RegexpParser — approximates constituency phrase structure
GRAMMAR = r"""
    ADJP: {<RB.*>?<JJ.*>+}
    NP:   {<DT>?<ADJP>?<NN.*>+}
          {<PRP>}
          {<NNP>+}
    VP:   {<MD>?<VB.*><NP|PP|ADJP>*}
    PP:   {<IN><NP>}
    S:    {<NP><VP>}
"""
chunk_parser = nltk.RegexpParser(GRAMMAR)


# 2A — CONSTITUENCY PARSING

print(f"\n\n{'─'*65}")
print("  2A — CONSTITUENCY PARSING  (NLTK RegexpParser)")
print(f"{'─'*65}")
print("""
  WHAT IS CONSTITUENCY PARSING?
  A constituency parse divides a sentence into nested phrases
  called constituents. Each node in the tree is a phrase type:

    S    = full Sentence
    NP   = Noun Phrase          e.g. "the brilliant director"
    VP   = Verb Phrase           e.g. "has a brilliant director"
    PP   = Prepositional Phrase  e.g. "near the river"
    ADJP = Adjective Phrase      e.g. "very interesting"

  Leaves of the tree are individual POS-tagged words.
  It answers: "What groups of words form meaningful phrases?"
""")

for i, sent in enumerate(sample_sentences, 1):
    tokens = word_tokenize(sent)
    tags   = pos_tag(tokens)        # POS-tag each token
    tree   = chunk_parser.parse(tags)  # build the constituency tree

    # Extract phrase types from the tree
    nps = [" ".join(w for w, t in sub.leaves())
           for sub in tree.subtrees() if sub.label() == "NP"]
    vps = [" ".join(w for w, t in sub.leaves())
           for sub in tree.subtrees() if sub.label() == "VP"]
    pps = [" ".join(w for w, t in sub.leaves())
           for sub in tree.subtrees() if sub.label() == "PP"]

    print(f"\n  -- Sentence {i}: \"{sent}\"")
    print(f"  POS Tags : {tags}")
    print(f"\n  Constituency Tree:")
    tree.pretty_print()                   # prints the indented tree diagram
    if nps: print(f"  Noun Phrases         : {nps}")
    if vps: print(f"  Verb Phrases         : {vps}")
    if pps: print(f"  Prepositional Phrases: {pps}")



# 2B — DEPENDENCY PARSING

print(f"\n\n{'─'*65}")
print("  2B — DEPENDENCY PARSING  (spaCy en_core_web_sm)")
print(f"{'─'*65}")
print("""
  WHAT IS DEPENDENCY PARSING?
  A dependency parse links every word directly to its grammatical
  HEAD word via a labeled arc (no phrase nodes, just word pairs).

  Common dependency labels:
    ROOT   = the main predicate of the sentence
    nsubj  = nominal subject       (who does the action)
    dobj   = direct object         (what receives the action)
    det    = determiner            (the, a)
    amod   = adjectival modifier   (adjective modifying a noun)
    advmod = adverbial modifier    (adverb modifying verb/adj)
    prep   = prepositional modifier
    pobj   = object of preposition
    aux    = auxiliary verb        (is, was, has)
    compound = compound noun part

  It answers: "What is the grammatical relationship between
               each pair of words in the sentence?"
""")

if SPACY_OK:
    for i, sent in enumerate(sample_sentences, 1):
        doc = nlp(sent)
        print(f"\n  -- Sentence {i}: \"{sent}\"")
        print(f"  {'Word':<18} {'POS':<8} {'Dep Label':<14} {'Head Word':<18} Children")
        print(f"  {'─'*17} {'─'*7} {'─'*13} {'─'*17} {'─'*20}")
        for token in doc:
            children = [c.text for c in token.children]
            print(f"  {token.text:<18} {token.pos_:<8} {token.dep_:<14}"
                  f" {token.head.text:<18} {children}")
else:
    # NLTK fallback — display POS tags when spaCy is unavailable
    for i, sent in enumerate(sample_sentences, 1):
        tokens = word_tokenize(sent)
        tags   = pos_tag(tokens)
        print(f"\n  -- Sentence {i}: \"{sent}\"")
        print(f"  POS Tags: {tags}")

# ── Detailed explanation using Sentence 1 as example ─────────────────────────
EXPLAIN_SENT = sample_sentences[0]
print(f"\n\n{'─'*65}")
print(f"  DETAILED EXPLANATION — Sentence 1 as Example")
print(f"{'─'*65}")
print(f"\n  Sentence: \"{EXPLAIN_SENT}\"")

# Build constituency tree for the explanation sentence
tokens_ex = word_tokenize(EXPLAIN_SENT)
tags_ex   = pos_tag(tokens_ex)
tree_ex   = chunk_parser.parse(tags_ex)
nps_ex    = [" ".join(w for w, t in sub.leaves())
             for sub in tree_ex.subtrees() if sub.label() == "NP"]
vps_ex    = [" ".join(w for w, t in sub.leaves())
             for sub in tree_ex.subtrees() if sub.label() == "VP"]

print(f"""
  CONSTITUENCY TREE EXPLANATION:
  The constituency parse breaks this sentence into nested phrases.
  The top node is S (Sentence) which contains NP + VP at minimum.

    NP (Noun Phrases) found : {nps_ex}
    VP (Verb Phrases) found : {vps_ex}

  Each NP or VP can itself contain smaller NPs, AdjPs and PPs,
  forming a TREE where the root is S and the leaves are words.
  This captures PHRASE STRUCTURE — which words form a unit together,
  without yet specifying the grammatical role each word plays.
""")

if SPACY_OK:
    # Build dependency tree for the explanation sentence
    doc_ex   = nlp(EXPLAIN_SENT)
    root_tok = [t for t in doc_ex if t.dep_ == "ROOT"][0]

    print(f"  DEPENDENCY TREE EXPLANATION:")
    print(f"  Every word points to its HEAD with a labeled relation.")
    print(f"  ROOT = \"{root_tok.text}\" — the main verb; all other words")
    print(f"  connect (directly or indirectly) back to it.\n")

    for token in doc_ex:
        if token.dep_ != "ROOT":
            arrow = f"\"{token.text}\"  --[{token.dep_}]-->  \"{token.head.text}\""
            print(f"    {arrow:<52}  ({explain_dep(token.dep_)})")

    print(f"""
  KEY DIFFERENCE BETWEEN THE TWO:
    Constituency tree  -> groups words into PHRASE NODES (NP, VP, PP ...)
                          Tells you WHAT kind of phrase each group forms.
    Dependency tree    -> links words via RELATION ARCS (nsubj, dobj ...)
                          Tells you WHO depends on WHOM and HOW.
  Both describe the same sentence from complementary structural angles.
""")


# ─────────────────────────────────────────────────────────────────────────────
#  PART 3 — NAMED ENTITY RECOGNITION (NER)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n\n{'─'*65}")
print("  Part 3: Named Entity Recognition (NER)")
print(f"{'─'*65}")

NER_ROWS = 200   # reviews to run NER on — set to len(df) for all 1000
print(f"\n  Running NER on {NER_ROWS} reviews ...")
print("  Entity types: PERSON, ORG, GPE/LOC, PRODUCT, DATE, WORK_OF_ART, EVENT\n")

entity_counts   = Counter()          # entity_type  → total count
entity_freq     = Counter()          # (text, type) → frequency across all reviews
entity_examples = defaultdict(list)  # entity_type  → up to 15 sample strings

entity_col = []   # one string per review storing all entities as "Text[TYPE]; ..."

if SPACY_OK:
    # spaCy NER — neural model, more accurate than NLTK rule-based chunker
    KEEP_TYPES = {"PERSON","ORG","GPE","LOC","PRODUCT",
                  "DATE","WORK_OF_ART","EVENT","NORP","FAC"}

    for idx, row in df.head(NER_ROWS).iterrows():
        # Run spaCy on the original (uncleaned) text for richer named entities
        doc          = nlp(str(row["review_text"]))
        row_entities = []

        for ent in doc.ents:
            if ent.label_ not in KEEP_TYPES:
                continue
            etext = ent.text.strip()
            etype = ent.label_

            entity_counts[etype] += 1
            entity_freq[(etext, etype)] += 1
            if len(entity_examples[etype]) < 15:
                entity_examples[etype].append(etext)
            row_entities.append(f"{etext}[{etype}]")

        entity_col.append("; ".join(row_entities))

else:
    # NLTK ne_chunk fallback when spaCy is unavailable
    NLTK_MAP = {"PERSON":"PERSON","ORGANIZATION":"ORG",
                "GPE":"GPE","LOCATION":"LOC","FACILITY":"FAC","GSP":"GPE"}

    for idx, row in df.head(NER_ROWS).iterrows():
        row_entities = []
        # Sentence-tokenize first so ne_chunk gets proper sentence boundaries
        for sent in sent_tokenize(str(row["review_text"])):
            chunked = ne_chunk(pos_tag(word_tokenize(sent)), binary=False)
            for subtree in chunked:
                if not isinstance(subtree, Tree):
                    continue
                etype = NLTK_MAP.get(subtree.label(), subtree.label())
                etext = " ".join(w for w, t in subtree.leaves())
                entity_counts[etype] += 1
                entity_freq[(etext, etype)] += 1
                if len(entity_examples[etype]) < 15:
                    entity_examples[etype].append(etext)
                row_entities.append(f"{etext}[{etype}]")
        entity_col.append("; ".join(row_entities))

# Pad remaining rows beyond NER_ROWS with empty string
while len(entity_col) < len(df):
    entity_col.append("")

df["named_entities"] = entity_col

# ── Entity count table ────────────────────────────────────────────────────────
total_ents = sum(entity_counts.values())

print(f"\n{'─'*65}")
print(f"  NER ENTITY COUNTS  (from {NER_ROWS} reviews)")
print(f"{'─'*65}")
print(f"\n  {'Entity Type':<18} {'Count':>8}   {'% of Total':>10}   Meaning")
print(f"  {'─'*17} {'─'*8}   {'─'*10}   {'─'*35}")

type_meanings = {
    "PERSON":     "People — actors, directors, characters",
    "ORG":        "Companies, studios, institutions",
    "GPE":        "Countries, cities, states",
    "LOC":        "Non-GPE locations (mountains, rivers)",
    "PRODUCT":    "Named products, objects, vehicles",
    "DATE":       "Dates, years, time periods",
    "WORK_OF_ART":"Titles of films, books, songs",
    "EVENT":      "Named events, wars, sports events",
    "NORP":       "Nationalities, religions, political groups",
    "FAC":        "Buildings, airports, highways",
}
for etype, cnt in sorted(entity_counts.items(), key=lambda x: -x[1]):
    pct     = cnt / total_ents * 100 if total_ents else 0
    meaning = type_meanings.get(etype, "")
    print(f"  {etype:<18} {cnt:>8,}   {pct:>9.1f}%   {meaning}")

print(f"  {'─'*17} {'─'*8}")
print(f"  {'TOTAL':<18} {total_ents:>8,}")

# ── Top 15 most frequent entities per type ────────────────────────────────────
print(f"\n{'─'*65}")
print("  TOP 15 MOST FREQUENT ENTITIES PER TYPE")
print(f"{'─'*65}")

for etype in ["PERSON","ORG","GPE","LOC","DATE","WORK_OF_ART","PRODUCT","EVENT","NORP"]:
    if etype not in entity_counts:
        continue
    # Filter the global frequency counter to this entity type
    top = [(txt, cnt) for (txt, t), cnt in entity_freq.most_common(500)
           if t == etype][:15]
    if not top:
        continue
    print(f"\n  [{etype}]  —  total occurrences: {entity_counts[etype]:,}")
    print(f"  {'Entity Text':<35} Count")
    print(f"  {'─'*34} {'─'*5}")
    for txt, cnt in top:
        print(f"  {txt:<35} {cnt:>5}")

# ── Sample reviews with entities shown ───────────────────────────────────────
print(f"\n{'─'*65}")
print("  SAMPLE REVIEWS WITH EXTRACTED ENTITIES")
print(f"{'─'*65}")

samples_shown = 0
for i, row in df.iterrows():
    if not row["named_entities"]:
        continue
    print(f"\n  Row {i+1} (sentiment: {row['sentiment']}):")
    print(f"  Text     : {str(row['review_text'])[:220]} ...")
    print(f"  Entities : {row['named_entities'][:300]}")
    samples_shown += 1
    if samples_shown >= SAMPLE_N:
        break


# ── Save final output CSV ─────────────────────────────────────────────────────
df.to_csv(OUTPUT_FILE, index=False)

print("\n\n" + "=" * 65)
print("  ANALYSIS COMPLETE  —  Summary")
print("=" * 65)

new_cols = [
    ("pos_noun_count", "Total nouns per review"),
    ("pos_verb_count", "Total verbs per review"),
    ("pos_adj_count",  "Total adjectives per review"),
    ("pos_adv_count",  "Total adverbs per review"),
    ("named_entities", "Extracted named entities"),
]
print(f"\n  {'New Column':<25} {'Non-empty':>10}   Description")
print(f"  {'─'*24} {'─'*10}   {'─'*35}")
for col, desc in new_cols:
    # For count columns check > 0, for text columns check string length > 0
    non_empty = df[col].astype(str).str.strip().gt("0").sum() if "count" in col \
                else df[col].astype(str).str.len().gt(0).sum()
    print(f"  {col:<25} {non_empty:>10,}   {desc}")

print(f"\n  Total POS tags counted  : {total_all:,}")
print(f"  Total named entities    : {total_ents:,}")
print(f"  Saved {len(df):,} rows → {OUTPUT_FILE}")
print(f"  All columns: {list(df.columns)}")
print("=" * 65)

  All NLTK resources ready.

  Loading spaCy model (en_core_web_sm) ...
  spaCy en_core_web_sm loaded.

  Loading data from: imdb_reviews_cleaned.csv
  Loaded 1,000 rows  |  Columns: ['movie_title', 'movie_imdb_id', 'movie_year', 'review_id', 'reviewer_username', 'rating', 'sentiment', 'review_title', 'review_text', 'review_date', 'helpful_votes', 'spoiler', 'review_url', 'collected_at', 'step1_no_special', 'step2_no_numbers', 'step3_lowercase', 'step4_no_stopwords', 'step5_stemmed', 'step6_lemmatized', 'clean_text']

  SYNTAX & STRUCTURE ANALYSIS  —  3 Parts

 Part 1: Parts-of-Speech (POS) Tagging

  Tagging 1,000 reviews ...

─────────────────────────────────────────────────────────────────
  SAMPLE POS TAGS  (first 3 reviews — first 20 tokens each)
─────────────────────────────────────────────────────────────────

  Row 1 (sentiment: positive):
  Text  : zentropa much common third man another noir like film set among rubble postwar europe like ttm much inventive camera work
  Tagged

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [9]:

import csv
import os
import re
import time
from collections import Counter
from datetime import datetime, timezone
from typing import Optional, List, Dict
from urllib.parse import urlparse, urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Stopwords (no downloads needed)
try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOPWORDS = set(ENGLISH_STOP_WORDS)
except Exception:
    STOPWORDS = set()


# Configuration
BASE_URL = "https://github.com/marketplace"
LISTING_URL = "https://github.com/marketplace?type=actions"

OUTPUT_RAW = "github_marketplace_actions.csv"
OUTPUT_CLEAN = "github_marketplace_actions_cleaned.csv"
QUALITY_REPORT = "github_marketplace_quality_report.txt"

TARGET_TOTAL = 1000
PAGE_DELAY = 2.0
MAX_RETRIES = 3
TIMEOUT = 30
MAX_EMPTY_PAGES = 3
ITEMS_PER_PAGE_EST = 20
MAX_PAGES = (TARGET_TOTAL // ITEMS_PER_PAGE_EST) + 30

CSV_COLUMNS = ["product_name", "description", "url", "page_number"]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

#  Lemmatization
USE_LEMMATIZER = False
if USE_LEMMATIZER:
    import nltk
    from nltk.stem import WordNetLemmatizer
    # nltk.download("wordnet")
    # nltk.download("omw-1.4")
    LEM = WordNetLemmatizer()

# Debug
DEBUG_CHECK_ACTIONS_IN_HTML = True  # prints if HTML contains "/marketplace/actions/"


def now_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def clean_cols(cols: List[str]) -> List[str]:
    return [c.replace("\ufeff", "").strip() for c in cols]


def name_from_url(u: str) -> str:
    slug = u.rstrip("/").split("/")[-1]
    return slug.replace("-", " ").strip()



# PART 1 — Scraper

def fetch_page(session: requests.Session, page_num: int) -> Optional[BeautifulSoup]:
    params = {"type": "actions", "page": page_num}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(BASE_URL, params=params, headers=HEADERS, timeout=TIMEOUT)

            if resp.status_code == 200:
                if DEBUG_CHECK_ACTIONS_IN_HTML:
                    has_actions = "/marketplace/actions/" in resp.text
                    print(f"(debug) actions-links-in-html={has_actions}", end=" ")
                return BeautifulSoup(resp.text, "html.parser")

            if resp.status_code == 429:
                wait = 30 * attempt
                print(f"    ⚠ Rate limited (429). Waiting {wait}s … (retry {attempt}/{MAX_RETRIES})")
                time.sleep(wait)
                continue

            if resp.status_code == 404:
                print(f"    Page {page_num} returned 404 — no more pages.")
                return None

            print(f"    HTTP {resp.status_code}. Retry {attempt}/{MAX_RETRIES}")
            time.sleep(PAGE_DELAY * attempt)

        except requests.exceptions.RequestException as exc:
            print(f"    Network error: {exc}. Retry {attempt}/{MAX_RETRIES}")
            time.sleep(PAGE_DELAY * attempt)

    print(f"    Failed after {MAX_RETRIES} retries on page {page_num}.")
    return None


def extract_description_from_anchor(a_tag, url: str, name: str) -> str:
    """
    FIX: Marketplace listing often stores the short description as text lines:
        [Name, "Action", Description, ...]
    We climb up to a parent that contains ONLY ONE action link and parse the text lines.
    """
    # Try up to N ancestors to find a “single-card” container
    candidate = a_tag
    for _ in range(1, 9):
        candidate = getattr(candidate, "parent", None)
        if candidate is None:
            break

        # Keep only containers that represent ONE listing card
        links = candidate.select("a[href*='/marketplace/actions/']")
        if len(links) != 1:
            continue

        lines = [s.strip() for s in candidate.stripped_strings if s.strip()]
        if not lines:
            continue

        # Typical pattern: Name -> Action -> Description
        for i, s in enumerate(lines):
            if s.lower() == "action" and i + 1 < len(lines):
                desc = lines[i + 1].strip()
                if desc and desc != name:
                    return desc

        # Fallback: first meaningful line after the name (skipping "Action")
        if name in lines:
            idx = lines.index(name)
            for s in lines[idx + 1:]:
                if s.lower() != "action" and s != name:
                    return s

    # Final fallback: no desc found
    return ""


def parse_products(soup: BeautifulSoup, page_num: int) -> List[Dict]:
    """
    Robust extraction:
    - Find ALL links containing '/marketplace/actions/'
    - Supports relative + absolute URLs
    - Deduplicate within page by URL
    - FIXED: extract description from card text lines (Name -> Action -> Description)
    """
    products: List[Dict] = []
    seen = set()

    for a in soup.select("a[href*='/marketplace/actions/']"):
        href = a.get("href", "")
        if not href:
            continue

        url = href if href.startswith("http") else urljoin("https://github.com", href)
        if url in seen:
            continue
        seen.add(url)

        name = a.get_text(strip=True) or name_from_url(url)
        description = extract_description_from_anchor(a, url, name)

        products.append({
            "product_name": name,
            "description": description,
            "url": url,
            "page_number": page_num
        })

    return products


def scrape_marketplace_actions() -> pd.DataFrame:
    print("=" * 72)
    print(" GitHub Marketplace Actions Scraper (PART 1)")
    print(f" Listing : {LISTING_URL}")
    print(f" Target  : {TARGET_TOTAL} unique products")
    print(f" Output  : {OUTPUT_RAW}")
    print("=" * 72)

    all_rows: List[Dict] = []
    seen_urls = set()
    empty_pages = 0

    with open(OUTPUT_RAW, "w", newline="", encoding="utf-8") as fout:
        writer = csv.DictWriter(fout, fieldnames=CSV_COLUMNS)
        writer.writeheader()

        with requests.Session() as session:
            for page_num in range(1, MAX_PAGES + 1):
                if len(all_rows) >= TARGET_TOTAL:
                    print(f"\n Reached target of {TARGET_TOTAL}. Stopping.")
                    break

                print(f"\nPage {page_num:>3} │ Fetching …", end=" ", flush=True)
                soup = fetch_page(session, page_num)
                if soup is None:
                    print("Failed / no more pages. Stopping.")
                    break

                products = parse_products(soup, page_num)

                new_products = []
                for p in products:
                    if p["url"] not in seen_urls:
                        seen_urls.add(p["url"])
                        new_products.append(p)

                if not new_products:
                    empty_pages += 1
                    print(f"0 new products (empty page #{empty_pages})")
                    if empty_pages >= MAX_EMPTY_PAGES:
                        print(f" {MAX_EMPTY_PAGES} consecutive empty pages — stopping.")
                        break
                    time.sleep(PAGE_DELAY)
                    continue

                empty_pages = 0

                remaining = TARGET_TOTAL - len(all_rows)
                if remaining <= 0:
                    break
                new_products = new_products[:remaining]

                writer.writerows(new_products)
                fout.flush()
                all_rows.extend(new_products)

                # small sanity check on first page
                if page_num == 1 and new_products:
                    sample = new_products[0]
                    print(f"+{len(new_products):>3} │ Total: {len(all_rows):>5} "
                          f"(sample desc len={len(sample['description'])})")
                else:
                    print(f"+{len(new_products):>3} │ Total: {len(all_rows):>5}")

                time.sleep(PAGE_DELAY)

    df = pd.DataFrame(all_rows, columns=CSV_COLUMNS)
    print("\n" + "=" * 72)
    print(" Scraping complete!")
    print(f" Total unique products collected : {len(df):,}")
    print(f" Raw CSV saved                  : {OUTPUT_RAW}")
    print("=" * 72)
    return df



# PART 2 — Data Quality + Preprocessing

REQUIRED_COLS = ["product_name", "description", "url", "page_number"]


def strip_html(text: str) -> str:
    return re.sub(r"<[^>]+>", " ", text)


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = strip_html(str(text))
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return normalize_whitespace(text)


def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text)


def remove_stopwords(tokens: List[str]) -> List[str]:
    if not STOPWORDS:
        return tokens
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


def lemmatize(tokens: List[str]) -> List[str]:
    if not USE_LEMMATIZER:
        return tokens
    return [LEM.lemmatize(t) for t in tokens]


def is_valid_action_url(u: str) -> bool:
    try:
        p = urlparse(u)
        return (p.netloc == "github.com") and (p.path.startswith("/marketplace/actions/"))
    except Exception:
        return False


def quality_and_preprocess(input_csv: str,
                           cleaned_csv: str = OUTPUT_CLEAN,
                           report_path: str = QUALITY_REPORT,
                           verbose: bool = True) -> pd.DataFrame:
    df = pd.read_csv(input_csv, encoding="utf-8")
    df.columns = clean_cols(list(df.columns))

    if df.empty:
        if verbose:
            print("\n[PART 2] Input CSV has 0 rows. Writing empty outputs.")
        for c in REQUIRED_COLS:
            if c not in df.columns:
                df[c] = pd.Series(dtype="object")
        df.to_csv(cleaned_csv, index=False, encoding="utf-8")
        with open(report_path, "w", encoding="utf-8") as f:
            f.write("Empty dataset. No preprocessing performed.\n")
        return df

    missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    input_rows = len(df)
    if verbose:
        print("\n[PART 2] Loaded raw CSV")
        print("Rows:", input_rows, "| Columns:", list(df.columns))
        print(df.head(3).to_string(index=False))

    for c in ["product_name", "description", "url"]:
        df[c] = df[c].astype(str).map(normalize_whitespace)

    df["page_number"] = pd.to_numeric(df["page_number"], errors="coerce").astype("Int64")
    if verbose:
        print("\n[PART 2] page_number numeric conversion done. Null:", int(df["page_number"].isna().sum()))

    before_url = len(df)
    df = df[df["url"].map(is_valid_action_url)].copy()
    invalid_url_removed = before_url - len(df)
    if verbose:
        print("\n[PART 2] URL validity filter:", "Before", before_url, "After", len(df), "Removed", invalid_url_removed)

    # Handle missing values
    df["product_name"] = df["product_name"].replace({"nan": ""}).fillna("")
    df["description"] = df["description"].replace({"nan": ""}).fillna("")
    df.loc[df["product_name"].str.len() == 0, "product_name"] = df["url"].map(name_from_url)

    before_dup = len(df)
    df = df.drop_duplicates(subset=["url"], keep="first").copy()
    dup_removed = before_dup - len(df)
    if verbose:
        print("\n[PART 2] Dedup by URL:", "Before", before_dup, "After", len(df), "Removed", dup_removed)

    # Preprocess descriptions
    df["description_clean"] = df["description"].map(clean_text)
    df["tokens"] = df["description_clean"].map(tokenize).map(remove_stopwords).map(lemmatize)
    df["token_count"] = df["tokens"].map(len)

    if verbose:
        print("\n[PART 2] Preprocessing sample:")
        print(df[["description", "description_clean", "tokens"]].head(3).to_string(index=False))

    df.to_csv(cleaned_csv, index=False, encoding="utf-8")

    token_counter = Counter(t for row in df["tokens"] for t in row)
    top10_tokens = token_counter.most_common(10)

    report_lines = []
    report_lines.append("GitHub Marketplace Actions — Data Quality & Preprocessing Report")
    report_lines.append("=" * 72)
    report_lines.append(f"Report generated at (UTC): {now_utc()}")
    report_lines.append("")
    report_lines.append("DATASET COUNTS")
    report_lines.append("-" * 72)
    report_lines.append(f"Input rows (raw CSV)                     : {input_rows}")
    report_lines.append(f"Invalid action-URL rows removed          : {invalid_url_removed}")
    report_lines.append(f"Duplicate URL rows removed               : {dup_removed}")
    report_lines.append(f"Final rows (cleaned dataset)             : {len(df)}")
    report_lines.append("")
    report_lines.append("MISSING VALUES (after cleaning)")
    report_lines.append("-" * 72)
    report_lines.append(df[REQUIRED_COLS].isna().sum().to_string())
    report_lines.append("")
    report_lines.append("TOKEN STATISTICS (description_clean)")
    report_lines.append("-" * 72)
    report_lines.append(df["token_count"].describe().to_string())
    report_lines.append("")
    report_lines.append("TOP 10 TOKENS")
    report_lines.append("-" * 72)
    for tok, freq in top10_tokens:
        report_lines.append(f"{tok:<20} {freq}")

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    print("\n" + "=" * 72)
    print(" Data Quality + Preprocessing complete! (PART 2)")
    print(f" Cleaned CSV saved  : {cleaned_csv}")
    print(f" Quality report     : {report_path}")
    print("=" * 72)

    return df


def preview_data(filepath: str, n: int = 5) -> pd.DataFrame:
    df = pd.read_csv(filepath, encoding="utf-8")
    df.columns = clean_cols(list(df.columns))
    print(f"\nDataset preview: {filepath}")
    print(f"Shape   : {df.shape[0]} rows × {df.shape[1]} cols")
    print(f"Columns : {list(df.columns)}")
    print("\nHead:")
    print(df.head(n).to_string(index=False))
    print("\nMissing values per column:")
    print(df.isnull().sum().to_string())
    return df



# MAIN — run end-to-end

if __name__ == "__main__":
    df_raw = scrape_marketplace_actions()

    if os.path.exists(OUTPUT_RAW):
        df_preview = preview_data(OUTPUT_RAW, n=5)

        # Quick check: ensure descriptions are actually captured
        desc_missing = int(pd.read_csv(OUTPUT_RAW)["description"].isna().sum())
        print(f"\nSanity check: description NaN count = {desc_missing} (should be near 0 now)")

        df_clean = quality_and_preprocess(OUTPUT_RAW, verbose=True)

        print("\nCleaned dataset preview:")
        print(df_clean.head(5).to_string(index=False))

 GitHub Marketplace Actions Scraper (PART 1)
 Listing : https://github.com/marketplace?type=actions
 Target  : 1000 unique products
 Output  : github_marketplace_actions.csv

Page   1 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:    20 (sample desc len=54)

Page   2 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:    40

Page   3 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:    60

Page   4 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:    80

Page   5 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:   100

Page   6 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:   120

Page   7 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:   140

Page   8 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:   160

Page   9 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:   180

Page  10 │ Fetching … (debug) actions-links-in-html=True + 20 │ Total:   200

Page  11 │ Fetching … (d

#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [10]:

import os
import re
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import pandas as pd
import tweepy



# CONFIG

BEARER_TOKEN = os.getenv("TWITTER_BEARER_TOKEN", "AAAAAAAAAAAAAAAAAAAAAF5A8AEAAAAAmIbproAEWUV5%2BP6UgPWj5zZ8xX8%3DRW0Du43XwMJHqq53hl4GMsOpscVOdxCzJ1IkXuWh8K6cFPbs4d")

HASHTAGS = ["#MachineLearning", "#ArtificialIntelligence", "#AI", "#DeepLearning", "#NLP"]
MAX_TWEETS = 500
LANG = "en"
INCLUDE_RETWEETS = False

RAW_CSV = "q5_tweets_raw.csv"
CLEAN_CSV = "q5_tweets_clean.csv"
QUALITY_REPORT = "q5_tweets_quality_report.txt"



# HELPERS

def now_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def require_bearer_token(token: str) -> None:
    if (not token) or ("PASTE_BEARER_TOKEN_HERE" in token):
        raise ValueError(
            "Bearer token missing. Set TWITTER_BEARER_TOKEN env var or paste it into BEARER_TOKEN."
        )


def build_query(hashtags: List[str], lang: str = "en", include_retweets: bool = False) -> str:
    joined = " OR ".join(hashtags)
    q = f"({joined})"
    if not include_retweets:
        q += " -is:retweet"
    if lang:
        q += f" lang:{lang}"
    return q


def clean_tweet_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"@\w+", " ", text)               # mentions
    text = re.sub(r"\s+", " ", text).strip()        # whitespace
    return text



# PART 1 — COLLECT

def collect_tweets_part1(bearer_token: str, query: str, max_tweets: int) -> Tuple[pd.DataFrame, Optional[str]]:
    """
    Returns (dataframe, error_message).
    If API fails (e.g., 402), df will be empty and error_message will be set.
    """
    require_bearer_token(bearer_token)

    client = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

    rows: List[Dict[str, str]] = []
    next_token: Optional[str] = None

    while len(rows) < max_tweets:
        remaining = max_tweets - len(rows)
        batch_size = 100 if remaining > 100 else remaining

        try:
            resp = client.search_recent_tweets(
                query=query,
                max_results=batch_size,
                tweet_fields=["id", "text", "author_id"],
                expansions=["author_id"],
                user_fields=["username"],
                next_token=next_token,
            )
        except tweepy.TooManyRequests:
            print("⚠ Rate limit hit. Sleeping 60 seconds...")
            time.sleep(60)
            continue
        except Exception as e:
            # Handles 402 Payment Required and others cleanly
            return pd.DataFrame(columns=["tweet_id", "username", "text"]), str(e)

        if not resp or not resp.data:
            break

        user_map: Dict[str, str] = {}
        if resp.includes and "users" in resp.includes:
            for u in resp.includes["users"]:
                user_map[str(u.id)] = u.username

        for t in resp.data:
            tweet_id = str(t.id)
            author_id = str(t.author_id) if t.author_id is not None else ""
            username = user_map.get(author_id, "")
            text = t.text or ""
            rows.append({"tweet_id": tweet_id, "username": username, "text": text})

        meta = resp.meta or {}
        next_token = meta.get("next_token")
        if not next_token:
            break

    return pd.DataFrame(rows, columns=["tweet_id", "username", "text"]), None



# PART 2 — CLEAN + DQ

def clean_and_quality_check_part2(df_raw: pd.DataFrame) -> Tuple[pd.DataFrame, str]:
    required_cols = ["tweet_id", "username", "text"]
    missing_cols = [c for c in required_cols if c not in df_raw.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    df = df_raw.copy()
    df["tweet_id"] = df["tweet_id"].astype(str)
    df["username"] = df["username"].astype(str)
    df["text"] = df["text"].astype(str)

    before_dup = len(df)
    df = df.drop_duplicates(subset=["tweet_id"], keep="first")
    dup_removed = before_dup - len(df)

    before_missing = len(df)
    df["tweet_id"] = df["tweet_id"].fillna("").str.strip()
    df["username"] = df["username"].fillna("").str.strip()
    df["text"] = df["text"].fillna("").str.strip()
    df = df[(df["tweet_id"] != "") & (df["text"] != "")]
    missing_removed = before_missing - len(df)

    df["text_clean"] = df["text"].map(clean_tweet_text)

    before_empty = len(df)
    df = df[df["text_clean"].str.len() > 0]
    empty_removed = before_empty - len(df)

    non_numeric_ids = int((~df["tweet_id"].str.match(r"^\d+$")).sum())
    empty_usernames = int((df["username"].str.len() == 0).sum())

    report = []
    report.append("Q5 Tweets — Data Quality Report")
    report.append("=" * 70)
    report.append(f"Generated at (UTC): {now_utc()}")
    report.append("")
    report.append("CLEANING + QUALITY SUMMARY")
    report.append("-" * 70)
    report.append(f"Rows (raw): {len(df_raw)}")
    report.append(f"Duplicates removed (tweet_id): {dup_removed}")
    report.append(f"Rows removed (missing tweet_id/text): {missing_removed}")
    report.append(f"Rows removed (empty cleaned text): {empty_removed}")
    report.append(f"Rows (cleaned): {len(df)}")
    report.append("")
    report.append("FINAL CONSISTENCY CHECKS")
    report.append("-" * 70)
    report.append(f"Non-numeric tweet_id count (should be 0): {non_numeric_ids}")
    report.append(f"Empty username count (allowed but noted): {empty_usernames}")
    report.append("")
    report.append("COMPLETENESS CHECK (cleaned)")
    report.append("-" * 70)
    report.append(df[["tweet_id", "username", "text_clean"]].isna().sum().to_string())

    return df, "\n".join(report)



# MAIN — END TO END

def main() -> None:
    query = build_query(HASHTAGS, lang=LANG, include_retweets=INCLUDE_RETWEETS)

    print("=" * 70)
    print("Q5 — Tweepy Tweet Scrape + Cleaning + Data Quality")
    print("Query:", query)
    print("Max tweets:", MAX_TWEETS)
    print("=" * 70)

    # PART 1
    print("\n[PART 1] Collecting tweets...")
    df_raw, api_error = collect_tweets_part1(BEARER_TOKEN, query, MAX_TWEETS)

    df_raw.to_csv(RAW_CSV, index=False, encoding="utf-8")
    print(f" Saved RAW CSV: {RAW_CSV} | rows={len(df_raw)}")

    # If API failed (e.g., 402), write a report and still produce empty clean CSV
    if api_error:
        print(f" API error: {api_error}")
        pd.DataFrame(columns=["tweet_id", "username", "text", "text_clean"]).to_csv(
            CLEAN_CSV, index=False, encoding="utf-8"
        )
        with open(QUALITY_REPORT, "w", encoding="utf-8") as f:
            f.write("Q5 Tweets — Data Quality Report\n")
            f.write("=" * 70 + "\n")
            f.write(f"Generated at (UTC): {now_utc()}\n\n")
            f.write("API request failed before data collection.\n")
            f.write(f"Error: {api_error}\n\n")
            f.write("Action required: Add X API credits in Developer Console (Pay-per-use).\n")
        print(f"Saved CLEAN CSV: {CLEAN_CSV}")
        print(f" Saved QUALITY REPORT: {QUALITY_REPORT}")
        return

    # PART 2
    print("\n[PART 2] Cleaning + final quality check...")
    if df_raw.empty:
        print("No tweets returned (no error). Possibly zero results for query/time window.")
        pd.DataFrame(columns=["tweet_id", "username", "text", "text_clean"]).to_csv(
            CLEAN_CSV, index=False, encoding="utf-8"
        )
        with open(QUALITY_REPORT, "w", encoding="utf-8") as f:
            f.write("No tweets returned; cleaned dataset is empty.\n")
        print(f"Saved CLEAN CSV: {CLEAN_CSV}")
        print(f" Saved QUALITY REPORT: {QUALITY_REPORT}")
        return

    df_clean, report = clean_and_quality_check_part2(df_raw)
    df_clean.to_csv(CLEAN_CSV, index=False, encoding="utf-8")
    with open(QUALITY_REPORT, "w", encoding="utf-8") as f:
        f.write(report)

    print(f" Saved CLEAN CSV: {CLEAN_CSV} | rows={len(df_clean)}")
    print(f" Saved QUALITY REPORT: {QUALITY_REPORT}")
    print("\nPreview (top 5 cleaned rows):")
    print(df_clean[["tweet_id", "username", "text_clean"]].head(5).to_string(index=False))


if __name__ == "__main__":
    main()

Q5 — Tweepy Tweet Scrape + Cleaning + Data Quality
Query: (#MachineLearning OR #ArtificialIntelligence OR #AI OR #DeepLearning OR #NLP) -is:retweet lang:en
Max tweets: 500

[PART 1] Collecting tweets...
 Saved RAW CSV: q5_tweets_raw.csv | rows=0
 API error: 402 Payment Required
Your enrolled account [2029749197768118275] does not have any credits to fulfill this request.
Saved CLEAN CSV: q5_tweets_clean.csv
 Saved QUALITY REPORT: q5_tweets_quality_report.txt


# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

This project was a great way to see a complete workflow from beginning to end: collecting the data, preprocessing it, checking the quality of the data, and exporting results for analytics. The biggest difficulty with this project was working with the real-world constraints of APIs: setting up authentication, rate limiting, pagination, and platform restrictions (e.g., the X API needs credits to be used, which can stop data collection even when the code is right); it was also a challenge to gracefully handle these failures and still produce valid outputs (i.e. raw/clean CSV file and a quality report).

What I especially liked was building a "clean" pipeline, one that is both reproducible and modular: gathering the data, applying consistent text cleaning (removing URLs and mentions, normalizing whitespace), and validating the dataset with completeness/consistency checks (i.e. duplicates, missing values, and ID formats). I found it rewarding to see the process produce formatted files so that I could begin analysis.

As far as the time that was provided for this task, the amount of time provided would be reasonable if everything went smoothly with API access, but it may become tight if there are outside factors that affect the external API (the tier of the API access, having credits, unexpected errors from the external API). I thought this was a fair project that was practical, but it is heavily reliant on the working API access being available from day one.